In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("SmartMeter")
         .master("local[*]")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

df = spark.read.parquet("C:/data/parquet_sample/")
df.printSchema()
print(df.count())
df.show(5)

C:\Users\hp\AppData\Roaming\Python\Python314\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Py4JJavaError: An error occurred while calling o29.parquet.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:817)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1415)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1620)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:802)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2079)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2123)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:1020)
	at org.apache.spark.util.HadoopFSUtils$.listLeafFiles(HadoopFSUtils.scala:247)
	at org.apache.spark.util.HadoopFSUtils$.$anonfun$parallelListLeafFilesInternal$1(HadoopFSUtils.scala:159)
	at scala.collection.immutable.List.map(L

In [1]:
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] += ";C:\\hadoop\\bin"

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("SmartMeter")
         .master("local[*]")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

df = spark.read.parquet("C:/data/parquet_sample/")
df.printSchema()
print(df.count())
df.show(5)

C:\Users\hp\AppData\Roaming\Python\Python314\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


root
 |-- LCLid: string (nullable = true)
 |-- stdorToU: string (nullable = true)
 |-- DateTime: timestamp_ntz (nullable = true)
 |-- KWH/hh (per half hour): string (nullable = true)
 |-- reading_date: date (nullable = true)
 |-- year: long (nullable = true)
 |-- month: long (nullable = true)
 |-- day: long (nullable = true)
 |-- hour: long (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- day_type: string (nullable = true)
 |-- half_hour: integer (nullable = true)
 |-- consumption_kwh: double (nullable = true)

4929099
+---------+--------+-------------------+----------------------+------------+----+-----+---+----+-----------+--------+---------+---------------+
|    LCLid|stdorToU|           DateTime|KWH/hh (per half hour)|reading_date|year|month|day|hour|day_of_week|day_type|half_hour|consumption_kwh|
+---------+--------+-------------------+----------------------+------------+----+-----+---+----+-----------+--------+---------+---------------+
|MAC004832|     ToU|2012-0

In [2]:
from pyspark.sql import functions as F

clean = (df.select(
            F.col("LCLid").alias("lclid"),
            F.col("stdorToU").alias("tariff"),
            F.col("DateTime").alias("reading_ts"),
            "reading_date", "year", "month", "day", "hour",
            "day_of_week", "day_type", "half_hour", "consumption_kwh")
         .dropna(subset=["lclid", "reading_ts", "consumption_kwh"])
         .dropDuplicates(["lclid", "reading_ts"]))

# 1. हर household का daily usage
daily = clean.groupBy("lclid", "tariff", "reading_date").agg(
    F.sum("consumption_kwh").alias("daily_kwh"),
    F.avg("consumption_kwh").alias("avg_kwh_hh"),
    F.max("consumption_kwh").alias("peak_kwh_hh"))

# 2. Tariff के हिसाब से hourly profile
hourly = clean.groupBy("tariff", "hour").agg(
    F.avg("consumption_kwh").alias("avg_kwh"))

# 3. Weekday vs Weekend
daytype = clean.groupBy("day_type", "tariff").agg(
    F.avg("consumption_kwh").alias("avg_kwh"))

daily.show(5)
hourly.orderBy("tariff", "hour").show(10)
daytype.show()

+---------+------+------------+------------------+-------------------+-----------+
|    lclid|tariff|reading_date|         daily_kwh|         avg_kwh_hh|peak_kwh_hh|
+---------+------+------------+------------------+-------------------+-----------+
|MAC004832|   ToU|  2013-06-25|             4.041|0.08418750000000001|      0.373|
|MAC004832|   ToU|  2013-08-12|7.8340000000000005|0.16320833333333334|      0.775|
|MAC004832|   ToU|  2013-09-24| 9.963000000000001|          0.2075625|      0.847|
|MAC004837|   ToU|  2012-05-04|             6.053|0.12610416666666666|      0.425|
|MAC004837|   ToU|  2012-06-10|             4.829|0.10060416666666666|      0.632|
+---------+------+------------+------------------+-------------------+-----------+
only showing top 5 rows
+------+----+-------------------+
|tariff|hour|            avg_kwh|
+------+----+-------------------+
|   ToU|   0| 0.1405674671550428|
|   ToU|   1|0.11145154248590539|
|   ToU|   2|0.10007274419761628|
|   ToU|   3|0.0961969663

In [3]:
clean.groupBy("tariff").count().show()

+------+-------+
|tariff|  count|
+------+-------+
|   ToU|4928948|
+------+-------+



In [4]:
out = "C:/data/output"
for name, d in [("clean", clean), ("daily", daily), ("hourly", hourly), ("daytype", daytype)]:
    (d.coalesce(1).write.mode("overwrite")
      .option("header", True).csv(f"{out}/{name}_csv"))